# 3_5 — Значимость и доверительные интервалы (P0-b)

Поверх pooled OOF из 3_4 (`cv_grouped_samples.csv`). torch не нужен.
- **Парный Wilcoxon signed-rank + Holm–Bonferroni** по per-sample ошибкам (post-prefix траектория, N_liq).
- **Object-cluster bootstrap** (кластеры = объекты/площадки) — корректные 95% CI для grouped CV вместо наивного 1.96·std/√n_folds; учитывает внутриобъектную корреляцию и site-shift.
- (Дополнительно) sample-bootstrap для сравнения.

In [1]:
import os, sys, json, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nick/Desktop/projects/liquefaction-ai


In [2]:
from liquefaction_ai.evaluation import paired_significance, object_cluster_bootstrap, bootstrap_classification
REF = 'DPI-Flow'
samples = pd.read_csv(os.path.join(TABLES,'cv_grouped_samples.csv'))
pair = paired_significance(samples, ref=REF)
pair.to_csv(os.path.join(TABLES,'significance_grouped_pairwise.csv'), index=False)
pair[['metric','model','n','median_diff','median_diff_lo','median_diff_hi','rank_biserial','p_holm','significant_0.05']].round(4)

,metric,model,n,median_diff,median_diff_lo,median_diff_hi,rank_biserial,p_holm,significant_0.05
0,traj_rmse_continuation,DPI-EVT,15,0.0021,0.0000,0.0155,0.8000,0.0419,True
1,traj_rmse_continuation,EVT-NeuralSSM,15,0.0163,0.0000,0.0260,0.7167,0.0789,False
2,traj_rmse_continuation,PINN,15,0.0000,-0.0035,0.0299,0.2333,1.0000,False
3,traj_rmse_continuation,Transformer,15,-0.0401,-0.0656,-0.0031,-1.0000,0.0051,True
4,traj_rmse_continuation,Neural Spline Flow,15,0.0000,-0.0037,0.0057,-0.0333,1.0000,False
5,traj_rmse_continuation,GRU,15,0.0000,-0.0109,0.0248,0.0833,1.0000,False
6,traj_rmse_continuation,TCN,15,0.0000,-0.0025,0.0598,0.4000,0.7279,False
7,nliq_log_err,DPI-EVT,16,-0.0002,-0.0216,0.0579,0.0882,1.0000,False
8,nliq_log_err,EVT-NeuralSSM,16,0.0141,-0.0113,0.2080,0.3824,0.5619,False
9,nliq_log_err,PINN,16,1.0265,0.5676,1.1703,1.0000,0.0039,True


## Object-cluster bootstrap (PRIMARY CI)

In [3]:
ocb = object_cluster_bootstrap(samples, ref=REF, nboot=2000)
ocb.to_csv(os.path.join(TABLES,'significance_grouped_cluster_bootstrap.csv'), index=False)
ocb

,model,AUROC,AUROC_lo,AUROC_hi,AUPRC,AUPRC_lo,AUPRC_hi,Brier,Brier_lo,Brier_hi,...,coverage90,coverage90_lo,coverage90_hi,AUROC_vs_ref_sig,AUPRC_vs_ref_sig,Brier_vs_ref_sig,ECE_vs_ref_sig,traj_rmse_continuation_vs_ref_sig,nliq_log_err_vs_ref_sig,coverage90_vs_ref_sig
0,DPI-Flow,0.9780,0.9515,0.9982,0.9781,0.9465,0.9991,0.0536,0.0146,0.1029,...,0.8387,0.7749,0.8968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DPI-EVT,0.9707,0.9427,0.9946,0.9719,0.9299,0.9969,0.0775,0.0269,0.1368,...,0.7938,0.6770,0.8836,False,False,False,False,True,False,False
2,EVT-NeuralSSM,0.9578,0.8998,0.9968,0.9596,0.9067,0.9981,0.0671,0.0183,0.1248,...,0.8178,0.7551,0.8814,False,False,False,False,False,False,False
3,PINN,0.8892,0.7581,0.9870,0.8709,0.7857,0.9932,0.1074,0.0475,0.1823,...,0.7684,0.6818,0.8478,True,True,True,False,False,True,True
4,Transformer,0.9263,0.8209,0.9983,0.9219,0.8658,0.9991,0.0615,0.0139,0.1335,...,0.9391,0.8865,0.9795,False,False,False,False,True,True,True
5,Neural Spline Flow,0.9771,0.9537,0.9957,0.9799,0.9429,0.9976,0.0545,0.0243,0.0900,...,0.7261,0.6684,0.7790,False,False,False,False,False,True,True
6,GRU,0.8593,0.7371,0.9562,0.8594,0.6524,0.9750,0.1424,0.0847,0.2043,...,0.8678,0.7737,0.9416,True,True,True,False,False,True,False
7,TCN,0.8742,0.7524,0.9661,0.8623,0.7581,0.9821,0.1259,0.0670,0.1939,...,0.7892,0.6991,0.8771,True,True,True,True,False,True,False
8,CatBoost,0.9943,0.9830,0.9991,0.9963,0.9867,0.9995,0.0327,0.0044,0.0725,...,NaN,NaN,NaN,True,False,False,False,False,False,False
